In [10]:
import pandas as pd
from pathlib import Path


In [11]:
transients = pd.read_parquet(
    "snia_results.parquet",
    engine="fastparquet",
)

In [12]:
transients

,diaObjectId,tns_name,z,catalogue_object_id,tns_prefix,type,ra,decl,absMag,nDiaSources
0,170019717277810735,2026ctw,0.150000,"10000,342494,11570",SN,SN Ia,151.740293,1.927064,NaN,754
1,170028486191218743,2026qrh,0.100000,"10000,81273,866",SN,SN Ia,64.185342,-48.943229,NaN,88
2,170028500610711681,2024xom,0.044167,CGCG036-024,SN,SN Ia,150.746125,2.336469,NaN,165
3,170028510695916335,2025mbp,0.046160,2MASXJ10062208+0109317,SN,SN Ia,151.592617,1.158385,NaN,273
4,170028520834072659,2026fgl,0.164000,WISEAJ095652.35+032811.7,SN,SN Ia,149.218165,3.469919,-16.5385,19
...,...,...,...,...,...,...,...,...,...,...
122,170661828398940941,2025bbx,0.067000,14470795-3930390,SN,SN Ia,221.785403,-39.509297,NaN,3
123,313893021499785278,2026ezw,0.131000,SDSSJ095832.59+011610.2,SN,SN Ia,149.635687,1.269344,NaN,20
124,313932566867476549,2026qrh,0.100000,"10000,81273,866",SN,SN Ia,64.185342,-48.943229,NaN,9
125,313972182793322538,2026fgl,0.164000,WISEAJ095652.35+032811.7,SN,SN Ia,149.218165,3.469919,-16.2020,17


In [13]:
transients = (
    transients[["diaObjectId", "tns_name", "ra", "decl"]]
    .rename(columns={"decl": "dec"})
    .dropna()
    .reset_index(drop=True)
    .copy()
)

transients["tns_name"] = transients["tns_name"].astype(str)

print(f"Associating {len(transients)} transients")
display(transients)

Associating 127 transients


,diaObjectId,tns_name,ra,dec
0,170019717277810735,2026ctw,151.740293,1.927064
1,170028486191218743,2026qrh,64.185342,-48.943229
2,170028500610711681,2024xom,150.746125,2.336469
3,170028510695916335,2025mbp,151.592617,1.158385
4,170028520834072659,2026fgl,149.218165,3.469919
...,...,...,...,...
122,170661828398940941,2025bbx,221.785403,-39.509297
123,313893021499785278,2026ezw,149.635687,1.269344
124,313932566867476549,2026qrh,64.185342,-48.943229
125,313972182793322538,2026fgl,149.218165,3.469919


In [14]:
import inspect
import textwrap
import astro_prost.helpers as helpers

func = helpers.build_panstarrs_candidates
source = textwrap.dedent(inspect.getsource(func))
lines = source.splitlines()

for i, line in enumerate(lines):
    if line.lstrip().startswith("temp_mag_r_std[temp_mag_r_std >"):
        indent = line[:len(line) - len(line.lstrip())]
        lines.insert(
            i,
            indent + "temp_mag_r_std = temp_mag_r_std.copy()",
        )
        break
else:
    raise RuntimeError("Could not locate the failing line; no patch applied.")

# Compile the patched function using the same module globals.
namespace = {}
exec(compile("\n".join(lines), "<prost_panstarrs_fix>", "exec"),
     helpers.__dict__, namespace)

# Update the existing function so existing references also use the fix.
func.__code__ = namespace[func.__name__].__code__

print("Applied writable-array fix to PROST's Pan-STARRS function.")

OSError: could not get source code

In [ ]:
import inspect
import textwrap
import traceback

import pandas as pd
import astro_prost.helpers as prost_helpers
from scipy.stats import gamma, uniform
from astro_prost import associate_sample

# Temporary read-only fix without editing helpers.py.
if not getattr(prost_helpers, "_shape_r_copy_patch_applied", False):
    source = textwrap.dedent(
        inspect.getsource(prost_helpers.calc_shape_props_decals)
    )

    old = 'candidate_hosts["shape_r"].values'
    new = 'candidate_hosts["shape_r"].to_numpy(copy=True)'

    if old not in source:
        raise RuntimeError("Expected shape_r code not found.")

    source = source.replace(old, new)
    exec(
        compile(source, "<prost_notebook_patch>", "exec"),
        prost_helpers.__dict__,
    )
    prost_helpers._shape_r_copy_patch_applied = True

# Collect results in memory.
results = []
failed = []

for i in range(len(transients)):
    one_transient = transients.iloc[[i]].copy().reset_index(drop=True)
    name = str(one_transient.iloc[0]["tns_name"])

    try:
        result = associate_sample(
        one_transient,
        catalogs=["decals", "panstarrs", "skymapper"],
        name_col="tns_name",
        coord_cols=("ra", "dec"),
        priors={"offset": uniform(loc=0, scale=10)},
        likes={"offset": gamma(a=0.75)},
        n_samples=1000,  # Match the 1,000 samples used internally.
        parallel=False,
        save=False,
        )
    

        if isinstance(result, Exception):
            raise result

        if not isinstance(result, pd.DataFrame):
            raise TypeError(
                f"Expected DataFrame; got {type(result).__name__}"
            )

        if result.empty:
            raise ValueError("PROST returned an empty DataFrame.")

        results.append(result)
        print(f"{i + 1}/{len(transients)} done: {name}")

    except Exception as exc:
        failed.append({"tns_name": name, "error": repr(exc)})
        print(f"{i + 1}/{len(transients)} failed: {name}: {exc}")
        traceback.print_exc()

hosts = pd.concat(results, ignore_index=True) if results else pd.DataFrame()
failed_df = pd.DataFrame(failed, columns=["tns_name", "error"])

display(hosts)
display(failed_df)

2026-10-14 11:21:43,280- INFO - Conditioning association on the following properties: ['offset']
2026-10-14 11:21:45,934- INFO - Loaded GLADE+ catalog.
2026-10-14 11:21:45,939- INFO - 

Associating 2026ctw at RA, DEC = 151.740293, 1.927064 (redshift nan)
2026-10-14 11:21:49,401- INFO - Association successful!
2026-10-14 11:21:49,421- INFO - 
    
Properties of best host (in decals dr9) for 2026ctw:
    objID: 9906622211952776
    R.A. (deg): 151.740166
    Dec. (deg): 1.927030
    Posterior: 1.0000e+00
    Redshift: 0.7213 ± 0.3376 (PHOT)
    Absolute Magnitude: -20.3 ± 1.1 (r)
    Transient Offset: 0.5 ± 0.1 arcsec
    Transient Offset Posterior: 1.0000e+00
2026-10-14 11:21:49,422- INFO - 
    
Properties of 2nd best host (in decals dr9) for 2026ctw:
    Redshift: 1.0042 ± 0.1648 (PHOT)
    Absolute Magnitude: -19.8 ± 1.2 (r)
    Transient Offset: 292.4 ± 0.1 arcsec
    Transient Offset Posterior: 0.0000e+00
2026-10-14 11:21:49,422- INFO - Chosen galaxy has catalog ID of 9906622211952

1/127 done: 2026ctw


2026-10-14 11:21:51,505- INFO - Loaded GLADE+ catalog.
2026-10-14 11:21:51,507- INFO - 

Associating 2026qrh at RA, DEC = 64.185342, -48.943229 (redshift nan)
2026-10-14 11:21:54,096- INFO - Association successful!
2026-10-14 11:21:54,214- INFO - 
    
Properties of best host (in decals dr9) for 2026qrh:
    objID: 9906605092569380
    R.A. (deg): 64.185370
    Dec. (deg): -48.943233
    Posterior: 9.8785e-01
    Redshift: 0.1125 ± 0.0167 (PHOT)
    Absolute Magnitude: -20.9 ± 0.9 (r)
    Transient Offset: 0.1 ± 0.1 arcsec
    Transient Offset Posterior: 9.8785e-01
2026-10-14 11:21:54,215- INFO - 
    
Properties of 2nd best host (in decals dr9) for 2026qrh:
    Redshift: 1.2030 ± 0.1423 (PHOT)
    Absolute Magnitude: -21.5 ± 1.2 (r)
    Transient Offset: 297.7 ± 0.1 arcsec
    Transient Offset Posterior: 0.0000e+00
2026-10-14 11:21:54,215- INFO - Chosen galaxy has catalog ID of 9906605092569380 and RA, DEC = 64.185370, -48.943233
2026-10-14 11:21:54,242- INFO - Conditioning associatio

2/127 done: 2026qrh


2026-10-14 11:21:56,405- INFO - Loaded GLADE+ catalog.
2026-10-14 11:21:56,406- INFO - 

Associating 2024xom at RA, DEC = 150.746125, 2.336469 (redshift nan)
2026-10-14 11:21:58,360- INFO - Association successful!
2026-10-14 11:21:58,385- INFO - 
    
Properties of best host (in decals dr9) for 2024xom:
    objID: 9906622306063947
    R.A. (deg): 150.743722
    Dec. (deg): 2.335828
    Posterior: 7.3434e-01
    Redshift: 0.1483 ± 0.0741 (PHOT)
    Absolute Magnitude: -20.1 ± 1.0 (r)
    Transient Offset: 8.9 ± 0.1 arcsec
    Transient Offset Posterior: 7.3434e-01
2026-10-14 11:21:58,386- INFO - 
    
Properties of 2nd best host (in decals dr9) for 2024xom:
    Redshift: 0.7043 ± 0.3117 (PHOT)
    Absolute Magnitude: -19.5 ± 1.2 (r)
    Transient Offset: 299.4 ± 0.1 arcsec
    Transient Offset Posterior: 0.0000e+00
2026-10-14 11:21:58,386- INFO - Chosen galaxy has catalog ID of 9906622306063947 and RA, DEC = 150.743722, 2.335828
2026-10-14 11:21:58,400- INFO - Conditioning association o

...

2026-10-14 11:29:08,056- INFO - Loaded GLADE+ catalog.
2026-10-14 11:29:08,058- INFO - 

Associating 2025imw at RA, DEC = 310.178614, -20.435643 (redshift nan)
2026-10-14 11:29:08,802- INFO - Running with shred_cut = True...
2026-10-14 11:29:14,852- INFO - Removing panstarrs dr2 shreds.
2026-10-14 11:29:14,986- INFO - Removed 23 flagged panstarrs sources.
2026-10-14 11:29:15,037- INFO - Association successful!
2026-10-14 11:29:15,054- INFO - 
    
Properties of best host (in panstarrs dr2) for 2025imw:
    objID: 83473101780730914
    R.A. (deg): 310.178090
    Dec. (deg): -20.441226
    Posterior: 4.3221e-01
    Redshift: 0.0000 ± 0.0000
    Absolute Magnitude: 0.0 ± 0.0
    Transient Offset: 20.2 ± 0.1 arcsec
    Transient Offset Posterior: 4.3221e-01
2026-10-14 11:29:15,055- INFO - 
    
Properties of 2nd best host (in panstarrs dr2) for 2025imw:
    objID: 83473101799986384
    R.A. (deg): 310.180010
    Dec. (deg): -20.436668
    Posterior: 3.4737e-01
    Redshift: 0.0000 ± 0.0000

74/127 done: 2025imw


2026-10-14 11:29:17,289- INFO - Loaded GLADE+ catalog.
2026-10-14 11:29:17,291- INFO - 

Associating 2025qaz at RA, DEC = 312.613622, -23.346686 (redshift nan)
2026-10-14 11:29:17,878- INFO - Running with shred_cut = True...
2026-10-14 11:29:21,664- INFO - Removing panstarrs dr2 shreds.
2026-10-14 11:29:21,785- INFO - Removed 1 flagged panstarrs sources.
2026-10-14 11:29:21,825- INFO - Association successful!
2026-10-14 11:29:21,840- INFO - 
    
Properties of best host (in panstarrs dr2) for 2025qaz:
    objID: 79983126136483689
    R.A. (deg): 312.613616
    Dec. (deg): -23.347225
    Posterior: 9.9976e-01
    Redshift: 0.0000 ± 0.0000
    Absolute Magnitude: 0.0 ± 0.0
    Transient Offset: 1.9 ± 0.1 arcsec
    Transient Offset Posterior: 9.9976e-01
2026-10-14 11:29:21,840- INFO - 
    
Properties of 2nd best host (in panstarrs dr2) for 2025qaz:
    Redshift: 0.0000 ± 0.0000
    Absolute Magnitude: 0.0 ± 0.0
    Transient Offset: 177.9 ± 0.1 arcsec
    Transient Offset Posterior: 0.0

75/127 done: 2025qaz


2026-10-14 11:29:25,282- INFO - Loaded GLADE+ catalog.
2026-10-14 11:29:25,297- INFO - 

Associating 2026onr at RA, DEC = 314.997109, -22.430263 (redshift nan)
2026-10-14 11:29:26,037- INFO - Running with shred_cut = True...


In [ ]:
results